# Method 3 — Stacking + GBDT Leaf Embeddings + MLP + Soft Retrieval

This corrected Method 3 keeps heterogeneous stacking **and restores tree-leaf embeddings**.

**RF + Extra Trees + XGBoost + CatBoost → leaf indices → one-hot leaf embeddings**

plus

**RF + Extra Trees + XGBoost + CatBoost + SVR → out-of-fold stacking predictions**

Then:

**Original features + leaf embeddings + stacking representation → Random ReLU → PCA → normalization → MLP → penultimate representation → soft k-NN retrieval → validation-selected α → final prediction.**

SVR remains in the stack but has no tree leaves, so it contributes through stacking predictions only.

# 1. Imports, Paths, and Reproducibility

All stochastic components use seed 42. The final test labels are not used for stacking or α selection.

In [1]:
import os
import time
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_squared_log_error
)
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPRegressor

import xgboost as xgb
from catboost import CatBoostRegressor

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

TRAIN_PATH = Path(r"E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_train.csv")
TEST_PATH = Path(r"E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_test.csv")

CHECKPOINT_DIR = Path(
    r"E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_4_v1\checkpoints"
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print("Train:", TRAIN_PATH)
print("Test :", TEST_PATH)
print("Checkpoint:", CHECKPOINT_DIR)
print("Seed:", RANDOM_SEED)


Train: E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_train.csv
Test : E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_test.csv
Checkpoint: E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_4_v1\checkpoints
Seed: 42


# 2. Load Dataset and Verify Target Columns

The actual dataset target names are `log_target` and `target_usd`.

In [2]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

TARGET_LOG = "log_target"
TARGET_USD = "target_usd"

for df, name in [(train_df, "train"), (test_df, "test")]:
    assert TARGET_LOG in df.columns, f"{TARGET_LOG} missing from {name}"
    assert TARGET_USD in df.columns, f"{TARGET_USD} missing from {name}"

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Log target :", TARGET_LOG)
print("USD target :", TARGET_USD)


Train shape: (16000, 81)
Test shape : (4000, 81)
Log target : log_target
USD target : target_usd


# 3. Feature Matrix and Targets

The two target columns are excluded from model inputs.

In [3]:
FEATURE_COLUMNS = [
    c for c in train_df.columns
    if c not in {TARGET_LOG, TARGET_USD}
]

X_all = train_df[FEATURE_COLUMNS].copy()
X_test_raw = test_df[FEATURE_COLUMNS].copy()

y_all_log = train_df[TARGET_LOG].to_numpy(dtype=np.float64)
y_test_log = test_df[TARGET_LOG].to_numpy(dtype=np.float64)

y_all_usd = train_df[TARGET_USD].to_numpy(dtype=np.float64)
y_test_usd = test_df[TARGET_USD].to_numpy(dtype=np.float64)

numeric_features = X_all.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [
    c for c in FEATURE_COLUMNS if c not in numeric_features
]

print("Features:", len(FEATURE_COLUMNS))
print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))


Features: 79
Numeric: 79
Categorical: 0


# 4. Train / Validation / Test Protocol

Twenty percent of the supplied training data is held out for validation. Validation selects α; the 4,000-row final test set remains untouched until final evaluation.

In [4]:
TRAIN_IDX, VAL_IDX = train_test_split(
    np.arange(len(X_all)),
    test_size=0.20,
    random_state=RANDOM_SEED,
    shuffle=True
)

X_train_raw = X_all.iloc[TRAIN_IDX].reset_index(drop=True)
X_val_raw = X_all.iloc[VAL_IDX].reset_index(drop=True)

y_train_log = y_all_log[TRAIN_IDX]
y_val_log = y_all_log[VAL_IDX]

y_train_usd = y_all_usd[TRAIN_IDX]
y_val_usd = y_all_usd[VAL_IDX]

print("Model-train:", len(TRAIN_IDX))
print("Validation :", len(VAL_IDX))
print("Final test :", len(test_df))


Model-train: 12800
Validation : 3200
Final test : 4000


# 5. Robust Preprocessing

The original processed feature path is retained. The preprocessor is fitted only on the model-training split.

In [5]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
        dtype=np.float32
    ))
])

robust_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

X_train_orig = np.asarray(
    robust_preprocessor.fit_transform(X_train_raw), dtype=np.float32
)
X_val_orig = np.asarray(
    robust_preprocessor.transform(X_val_raw), dtype=np.float32
)
X_test_orig = np.asarray(
    robust_preprocessor.transform(X_test_raw), dtype=np.float32
)

print("Processed train:", X_train_orig.shape)
print("Processed val  :", X_val_orig.shape)
print("Processed test :", X_test_orig.shape)


Processed train: (12800, 79)
Processed val  : (3200, 79)
Processed test : (4000, 79)


# 6. Define the Stacking Base Learners

The replacement for the single XGBoost feature generator is a heterogeneous ensemble of Random Forest, Extra Trees, XGBoost, CatBoost, and SVR. These are deliberately controlled rather than individually tuned.

In [6]:
BASE_MODELS = {
    "random_forest": RandomForestRegressor(
        n_estimators=150,
        max_depth=12,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "extra_trees": ExtraTreesRegressor(
        n_estimators=150,
        max_depth=None,
        min_samples_leaf=2,
        max_features="sqrt",
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "xgboost": xgb.XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=1.0,
        colsample_bytree=1.0,
        min_child_weight=1,
        reg_lambda=1.0,
        reg_alpha=0.0,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "catboost": CatBoostRegressor(
        iterations=100,
        depth=6,
        learning_rate=0.1,
        loss_function="RMSE",
        random_seed=RANDOM_SEED,
        verbose=False,
        thread_count=-1
    ),

    "svr": SVR(
        kernel="rbf",
        C=10.0,
        epsilon=0.1,
        gamma="scale"
    )
}

print("Base learners:", list(BASE_MODELS))


Base learners: ['random_forest', 'extra_trees', 'xgboost', 'catboost', 'svr']


# 7. Leakage-Controlled Out-of-Fold Stacking

This is the critical stacking stage.

For every model-training row, the base-model prediction is generated by a model that did not train on that row. These out-of-fold predictions become the training stack representation.

For validation and test, each base learner is refit on all model-training rows and predicts the unseen split. This prevents in-sample base predictions from leaking into the downstream MLP.

In [7]:
N_STACK_FOLDS = 5
kf = KFold(
    n_splits=N_STACK_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED
)

model_names = list(BASE_MODELS.keys())

stack_train = np.zeros(
    (len(X_train_orig), len(model_names)), dtype=np.float32
)
stack_val = np.zeros(
    (len(X_val_orig), len(model_names)), dtype=np.float32
)
stack_test = np.zeros(
    (len(X_test_orig), len(model_names)), dtype=np.float32
)

fitted_base_models = {}
base_training_times = {}

stack_start = time.perf_counter()

for col, name in enumerate(model_names):
    template = BASE_MODELS[name]
    oof = np.zeros(len(X_train_orig), dtype=np.float64)
    model_start = time.perf_counter()

    for fold, (fit_idx, holdout_idx) in enumerate(
        kf.split(X_train_orig), start=1
    ):
        model = clone(template)
        model.fit(X_train_orig[fit_idx], y_train_log[fit_idx])
        oof[holdout_idx] = model.predict(X_train_orig[holdout_idx])
        print(f"{name:15s} | fold {fold}/{N_STACK_FOLDS}")

    stack_train[:, col] = oof.astype(np.float32)

    full_model = clone(template)
    full_model.fit(X_train_orig, y_train_log)

    stack_val[:, col] = full_model.predict(X_val_orig).astype(np.float32)
    stack_test[:, col] = full_model.predict(X_test_orig).astype(np.float32)

    fitted_base_models[name] = full_model
    base_training_times[name] = time.perf_counter() - model_start

stack_base_training_time = time.perf_counter() - stack_start

print("\nStack shapes:")
print("Train:", stack_train.shape)
print("Val  :", stack_val.shape)
print("Test :", stack_test.shape)
print("\nTraining times:")
for name, seconds in base_training_times.items():
    print(f"{name:15s}: {seconds:.3f}s")
print("Total stacking stage:", stack_base_training_time)


random_forest   | fold 1/5
random_forest   | fold 2/5
random_forest   | fold 3/5
random_forest   | fold 4/5
random_forest   | fold 5/5
extra_trees     | fold 1/5
extra_trees     | fold 2/5
extra_trees     | fold 3/5
extra_trees     | fold 4/5
extra_trees     | fold 5/5
xgboost         | fold 1/5
xgboost         | fold 2/5
xgboost         | fold 3/5
xgboost         | fold 4/5
xgboost         | fold 5/5
catboost        | fold 1/5
catboost        | fold 2/5
catboost        | fold 3/5
catboost        | fold 4/5
catboost        | fold 5/5
svr             | fold 1/5
svr             | fold 2/5
svr             | fold 3/5
svr             | fold 4/5
svr             | fold 5/5

Stack shapes:
Train: (12800, 5)
Val  : (3200, 5)
Test : (4000, 5)

Training times:
random_forest  : 4.095s
extra_trees    : 4.410s
xgboost        : 2.199s
catboost       : 3.253s
svr            : 78.910s
Total stacking stage: 92.86812259999999


# 8. Ridge Stacking Meta-Learner

A Ridge meta-learner combines the five out-of-fold base predictions. The five individual predictions plus the meta prediction form the stacking representation.

The next stage additionally extracts tree leaf indices from the fitted Random Forest, Extra Trees, XGBoost, and CatBoost models.

In [8]:
meta_learner = Ridge(alpha=1.0)

meta_start = time.perf_counter()
meta_learner.fit(stack_train, y_train_log)
meta_training_time = time.perf_counter() - meta_start

meta_train_pred = meta_learner.predict(stack_train).astype(np.float32)
meta_val_pred = meta_learner.predict(stack_val).astype(np.float32)
meta_test_pred = meta_learner.predict(stack_test).astype(np.float32)

STACK_REP_TRAIN = np.hstack([stack_train, meta_train_pred[:, None]])
STACK_REP_VAL = np.hstack([stack_val, meta_val_pred[:, None]])
STACK_REP_TEST = np.hstack([stack_test, meta_test_pred[:, None]])

print("Stack representation train:", STACK_REP_TRAIN.shape)
print("Stack representation val  :", STACK_REP_VAL.shape)
print("Stack representation test :", STACK_REP_TEST.shape)

print("\nMeta coefficients:")
for name, coef in zip(model_names, meta_learner.coef_):
    print(f"{name:15s}: {coef:.6f}")


Stack representation train: (12800, 6)
Stack representation val  : (3200, 6)
Stack representation test : (4000, 6)

Meta coefficients:
random_forest  : -0.225021
extra_trees    : 0.213970
xgboost        : 0.683774
catboost       : 0.395961
svr            : -0.037061


# 9. Tree Leaf Indices → One-Hot Leaf Embeddings

This restores the missing tree/GBDT embedding component.

Each fitted tree ensemble provides one leaf index per tree for every sample:
- Random Forest
- Extra Trees
- XGBoost
- CatBoost

The leaf-index matrix is then one-hot encoded.

SVR is excluded from this stage because it is not a tree model and has no leaf indices. Its information is already represented by the stacking predictions.

The leaf encoder is fitted on the model-training split and then applied to validation/test.

In [9]:
# ============================================================
# TREE LEAF INDICES → SPARSE ONE-HOT LEAF EMBEDDINGS
# ============================================================

from sklearn.preprocessing import OneHotEncoder
import numpy as np

TREE_MODEL_NAMES = [
    "random_forest",
    "extra_trees",
    "xgboost",
    "catboost"
]


def get_leaf_indices(model_name, model, X):
    """
    Extract one leaf index per tree for every sample.

    Output shape:
        (n_samples, n_trees)
    """

    if model_name in ("random_forest", "extra_trees"):
        return np.asarray(
            model.apply(X),
            dtype=np.int64
        )

    elif model_name == "xgboost":
        return np.asarray(
            model.apply(X),
            dtype=np.int64
        )

    elif model_name == "catboost":
        return np.asarray(
            model.calc_leaf_indexes(X),
            dtype=np.int64
        )

    else:
        raise ValueError(
            f"Unsupported tree model: {model_name}"
        )


# ------------------------------------------------------------
# 1. Extract leaf indices
# ------------------------------------------------------------

leaf_train_parts = []
leaf_val_parts = []
leaf_test_parts = []

for name in TREE_MODEL_NAMES:

    model = fitted_base_models[name]

    print(f"\nExtracting leaf indices: {name}")

    train_leaf = get_leaf_indices(
        name,
        model,
        X_train_orig
    )

    val_leaf = get_leaf_indices(
        name,
        model,
        X_val_orig
    )

    test_leaf = get_leaf_indices(
        name,
        model,
        X_test_orig
    )

    # --------------------------------------------------------
    # Keep each tree/model's leaf-ID space separate.
    #
    # We use the TRAINING leaf IDs to define offsets.
    # --------------------------------------------------------

    max_leaf = train_leaf.max(axis=0)

    offsets = np.cumsum(
        np.r_[0, max_leaf[:-1] + 1]
    ).astype(np.int64)

    train_leaf = train_leaf + offsets
    val_leaf = val_leaf + offsets
    test_leaf = test_leaf + offsets

    leaf_train_parts.append(train_leaf)
    leaf_val_parts.append(val_leaf)
    leaf_test_parts.append(test_leaf)

    print(
        f"{name:15s} | "
        f"n_trees={train_leaf.shape[1]:4d} | "
        f"train={train_leaf.shape} | "
        f"val={val_leaf.shape} | "
        f"test={test_leaf.shape}"
    )


# ------------------------------------------------------------
# 2. Combine leaf indices from all tree models
# ------------------------------------------------------------

LEAF_IDS_TRAIN = np.hstack(leaf_train_parts)
LEAF_IDS_VAL   = np.hstack(leaf_val_parts)
LEAF_IDS_TEST  = np.hstack(leaf_test_parts)

print("\n" + "=" * 70)
print("COMBINED LEAF-INDEX MATRICES")
print("=" * 70)

print("Train:", LEAF_IDS_TRAIN.shape)
print("Val  :", LEAF_IDS_VAL.shape)
print("Test :", LEAF_IDS_TEST.shape)


# ------------------------------------------------------------
# 3. One-hot encode leaf indices AS SPARSE MATRICES
# ------------------------------------------------------------
#
# IMPORTANT:
# sparse_output=True
#
# DO NOT change this to False.
#
# The leaf embedding may contain hundreds of thousands
# of columns but only a small number of non-zero values
# per sample.
# ------------------------------------------------------------

leaf_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True,
    dtype=np.float32
)


print("\nFitting leaf encoder on TRAINING data...")

LEAF_EMBED_TRAIN = leaf_encoder.fit_transform(
    LEAF_IDS_TRAIN
)

print("Transforming validation data...")

LEAF_EMBED_VAL = leaf_encoder.transform(
    LEAF_IDS_VAL
)

print("Transforming test data...")

LEAF_EMBED_TEST = leaf_encoder.transform(
    LEAF_IDS_TEST
)


# ------------------------------------------------------------
# 4. Display embedding dimensions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPARSE ONE-HOT LEAF EMBEDDINGS")
print("=" * 70)

print("Train:", LEAF_EMBED_TRAIN.shape)
print("Val  :", LEAF_EMBED_VAL.shape)
print("Test :", LEAF_EMBED_TEST.shape)


# ------------------------------------------------------------
# 5. Check sparsity
# ------------------------------------------------------------

print("\nNumber of non-zero values:")

print(
    "Train:",
    LEAF_EMBED_TRAIN.nnz
)

print(
    "Val  :",
    LEAF_EMBED_VAL.nnz
)

print(
    "Test :",
    LEAF_EMBED_TEST.nnz
)


# ------------------------------------------------------------
# 6. Estimate actual sparse memory usage
# ------------------------------------------------------------

def sparse_memory_mb(matrix):
    return (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    ) / (1024 ** 2)


print("\nApproximate sparse memory usage:")

print(
    f"Train: {sparse_memory_mb(LEAF_EMBED_TRAIN):.2f} MB"
)

print(
    f"Val  : {sparse_memory_mb(LEAF_EMBED_VAL):.2f} MB"
)

print(
    f"Test : {sparse_memory_mb(LEAF_EMBED_TEST):.2f} MB"
)


# ------------------------------------------------------------
# 7. Sanity check
# ------------------------------------------------------------

assert LEAF_EMBED_TRAIN.shape[0] == len(X_train_orig)
assert LEAF_EMBED_VAL.shape[0] == len(X_val_orig)
assert LEAF_EMBED_TEST.shape[0] == len(X_test_orig)

print("\n✓ Leaf-index extraction completed successfully.")
print("✓ One-hot leaf embeddings are stored as SPARSE matrices.")
print("✓ No dense 622k-column matrix was created.")


Extracting leaf indices: random_forest
random_forest   | n_trees= 150 | train=(12800, 150) | val=(3200, 150) | test=(4000, 150)

Extracting leaf indices: extra_trees
extra_trees     | n_trees= 150 | train=(12800, 150) | val=(3200, 150) | test=(4000, 150)

Extracting leaf indices: xgboost
xgboost         | n_trees= 100 | train=(12800, 100) | val=(3200, 100) | test=(4000, 100)

Extracting leaf indices: catboost
catboost        | n_trees= 100 | train=(12800, 100) | val=(3200, 100) | test=(4000, 100)

COMBINED LEAF-INDEX MATRICES
Train: (12800, 500)
Val  : (3200, 500)
Test : (4000, 500)

Fitting leaf encoder on TRAINING data...
Transforming validation data...
Transforming test data...

SPARSE ONE-HOT LEAF EMBEDDINGS
Train: (12800, 622279)
Val  : (3200, 622279)
Test : (4000, 622279)

Number of non-zero values:
Train: 6400000
Val  : 1599940
Test : 1999919

Approximate sparse memory usage:
Train: 48.88 MB
Val  : 12.22 MB
Test : 15.27 MB

✓ Leaf-index extraction completed successfully.
✓ One-

# 10. Concatenate Original Features + Leaf Embeddings + Stacking Representation

This is the corrected Proposed 3 representation:

**Original preprocessed features + one-hot tree leaf embeddings + heterogeneous stacking representation.**

In [10]:
from scipy.sparse import csr_matrix, hstack

Psi_train = hstack([
    csr_matrix(X_train_orig, dtype=np.float32),
    LEAF_EMBED_TRAIN,
    csr_matrix(STACK_REP_TRAIN, dtype=np.float32)
], format="csr")

Psi_val = hstack([
    csr_matrix(X_val_orig, dtype=np.float32),
    LEAF_EMBED_VAL,
    csr_matrix(STACK_REP_VAL, dtype=np.float32)
], format="csr")

Psi_test = hstack([
    csr_matrix(X_test_orig, dtype=np.float32),
    LEAF_EMBED_TEST,
    csr_matrix(STACK_REP_TEST, dtype=np.float32)
], format="csr")

print("Combined representation:")
print("Train:", Psi_train.shape)
print("Val  :", Psi_val.shape)
print("Test :", Psi_test.shape)

Combined representation:
Train: (12800, 622364)
Val  : (3200, 622364)
Test : (4000, 622364)


# 11. Random Feature Expansion + ReLU

The random feature stage follows the same paper-inspired Gaussian scaling and ReLU transformation used in the previous method. citeturn0view0

In [11]:
RANDOM_FEATURES = 1024
input_dim = Psi_train.shape[1]

Omega = rng.normal(
    loc=0.0,
    scale=np.sqrt(2.0 / RANDOM_FEATURES),
    size=(input_dim, RANDOM_FEATURES)
).astype(np.float32)

def random_relu_features(X, Omega):
    return np.maximum(X @ Omega, 0.0).astype(np.float32)

R_train = random_relu_features(Psi_train, Omega)
R_val = random_relu_features(Psi_val, Omega)
R_test = random_relu_features(Psi_test, Omega)

print("Random-ReLU train:", R_train.shape)
print("Random-ReLU val  :", R_val.shape)
print("Random-ReLU test :", R_test.shape)


Random-ReLU train: (12800, 1024)
Random-ReLU val  : (3200, 1024)
Random-ReLU test : (4000, 1024)


# 12. PCA Fixed-Size Projection

PCA is fitted only on the model-training random features and then applied to validation and test.

In [12]:
MAIN_DIM = min(128, R_train.shape[1], R_train.shape[0] - 1)

pca = PCA(
    n_components=MAIN_DIM,
    random_state=RANDOM_SEED
)

Z_train_pca = pca.fit_transform(R_train).astype(np.float32)
Z_val_pca = pca.transform(R_val).astype(np.float32)
Z_test_pca = pca.transform(R_test).astype(np.float32)

print("PCA dimension:", MAIN_DIM)
print("Explained variance:", pca.explained_variance_ratio_.sum())
print("PCA train:", Z_train_pca.shape)
print("PCA val  :", Z_val_pca.shape)
print("PCA test :", Z_test_pca.shape)


PCA dimension: 128
Explained variance: 0.96422416
PCA train: (12800, 128)
PCA val  : (3200, 128)
PCA test : (4000, 128)


# 13. Feature-Wise Normalization

Training PCA statistics are reused unchanged for validation and test.

In [13]:
norm_mean = Z_train_pca.mean(axis=0)
norm_std = Z_train_pca.std(axis=0)

def normalize_representation(Z):
    return (
        (Z - norm_mean) /
        np.sqrt(norm_std ** 2 + 1e-8)
    ).astype(np.float32)

Z_train = normalize_representation(Z_train_pca)
Z_val = normalize_representation(Z_val_pca)
Z_test = normalize_representation(Z_test_pca)

print("Normalized shape:", Z_train.shape)
print("Training mean abs:", np.mean(np.abs(Z_train.mean(axis=0))))
print("Training mean std:", np.mean(Z_train.std(axis=0)))


Normalized shape: (12800, 128)
Training mean abs: 1.3924346e-08
Training mean std: 0.9999999


# 14. MLP Main Network

The MLP predicts `log_target`. Its penultimate hidden representation is retained for retrieval, matching the downstream structure we froze earlier.

In [14]:
MLP_HIDDEN_LAYERS = (512, 512, 512)

mlp = MLPRegressor(
    hidden_layer_sizes=MLP_HIDDEN_LAYERS,
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=128,
    learning_rate_init=1e-3,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=20,
    random_state=RANDOM_SEED,
    verbose=True
)

mlp_start = time.perf_counter()
mlp.fit(Z_train, y_train_log)
mlp_training_time = time.perf_counter() - mlp_start

y_train_main_log = mlp.predict(Z_train)
y_val_main_log = mlp.predict(Z_val)
y_test_main_log = mlp.predict(Z_test)

print("MLP training time:", mlp_training_time)
print("MLP iterations:", mlp.n_iter_)


Iteration 1, loss = 4.10648739
Validation score: 0.245978
Iteration 2, loss = 2.63524218
Validation score: 0.359326
Iteration 3, loss = 2.29859894
Validation score: 0.345991
Iteration 4, loss = 1.96058546
Validation score: 0.338124
Iteration 5, loss = 1.59554043
Validation score: 0.286835
Iteration 6, loss = 1.13845615
Validation score: 0.260769
Iteration 7, loss = 0.76244094
Validation score: 0.237265
Iteration 8, loss = 0.51247141
Validation score: 0.225676
Iteration 9, loss = 0.30615443
Validation score: 0.203260
Iteration 10, loss = 0.18908065
Validation score: 0.224745
Iteration 11, loss = 0.12459974
Validation score: 0.231227
Iteration 12, loss = 0.10124516
Validation score: 0.226992
Iteration 13, loss = 0.09627852
Validation score: 0.215699
Iteration 14, loss = 0.09416118
Validation score: 0.232367
Iteration 15, loss = 0.10109024
Validation score: 0.223832
Iteration 16, loss = 0.09679933
Validation score: 0.220246
Iteration 17, loss = 0.09191836
Validation score: 0.213470
Iterat

# 15. Extract Penultimate MLP Representation

Retrieval operates on the penultimate MLP representation rather than on the final prediction.

In [15]:
def relu(x):
    return np.maximum(x, 0.0)

def mlp_penultimate_transform(model, X):
    A = np.asarray(X, dtype=np.float32)
    for layer_idx in range(len(model.coefs_) - 1):
        A = A @ model.coefs_[layer_idx] + model.intercepts_[layer_idx]
        A = relu(A)
    return np.asarray(A, dtype=np.float32)

H_train = mlp_penultimate_transform(mlp, Z_train)
H_val = mlp_penultimate_transform(mlp, Z_val)
H_test = mlp_penultimate_transform(mlp, Z_test)

print("Penultimate train:", H_train.shape)
print("Penultimate val  :", H_val.shape)
print("Penultimate test :", H_test.shape)


Penultimate train: (12800, 512)
Penultimate val  : (3200, 512)
Penultimate test : (4000, 512)


# 16. Soft k-NN Retrieval

Retrieval uses cosine similarity, temperature scaling, and a weighted average of training `log_target` values. Only the model-training split is used as retrieval context.

In [16]:
RETRIEVAL_K = 64
RETRIEVAL_TEMPERATURE = 0.10

def l2_normalize_rows(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, 1e-12)

H_train_norm = l2_normalize_rows(H_train).astype(np.float32)
H_val_norm = l2_normalize_rows(H_val).astype(np.float32)
H_test_norm = l2_normalize_rows(H_test).astype(np.float32)

retrieval_index = NearestNeighbors(
    n_neighbors=min(RETRIEVAL_K, len(H_train_norm)),
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)
retrieval_index.fit(H_train_norm)

def soft_retrieval(query_rep, context_y):
    distances, indices = retrieval_index.kneighbors(
        query_rep,
        n_neighbors=min(RETRIEVAL_K, len(H_train_norm))
    )

    similarities = 1.0 - distances
    logits = similarities / RETRIEVAL_TEMPERATURE
    logits -= logits.max(axis=1, keepdims=True)

    weights = np.exp(logits)
    weights /= weights.sum(axis=1, keepdims=True)

    retrieved_y = context_y[indices]
    prediction = np.sum(weights * retrieved_y, axis=1)

    return prediction, similarities, weights, indices

y_val_retrieval_log, val_similarities, val_weights, val_indices = soft_retrieval(
    H_val_norm, y_train_log
)

y_test_retrieval_log, test_similarities, test_weights, test_indices = soft_retrieval(
    H_test_norm, y_train_log
)

print("Validation retrieval:", y_val_retrieval_log.shape)
print("Test retrieval:", y_test_retrieval_log.shape)


Validation retrieval: (3200,)
Test retrieval: (4000,)


# 17. Validation Selection of α

The final combination is `(1 - α) * main + α * retrieval`. α is selected only from validation RMSE_log and then frozen before test evaluation. citeturn0view0

In [17]:
ALPHA_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

alpha_rows = []

for alpha in ALPHA_GRID:
    pred = (
        (1.0 - alpha) * y_val_main_log
        + alpha * y_val_retrieval_log
    )

    alpha_rows.append({
        "alpha": alpha,
        "MAE_log": mean_absolute_error(y_val_log, pred),
        "MSE_log": mean_squared_error(y_val_log, pred),
        "RMSE_log": np.sqrt(mean_squared_error(y_val_log, pred)),
        "R2_log": r2_score(y_val_log, pred)
    })

alpha_validation_results = (
    pd.DataFrame(alpha_rows)
    .sort_values("RMSE_log")
    .reset_index(drop=True)
)

best_alpha = float(alpha_validation_results.loc[0, "alpha"])

display(alpha_validation_results)
print("Selected alpha:", best_alpha)


,alpha,MAE_log,MSE_log,RMSE_log,R2_log
0,0.2,1.768673,5.597384,2.365879,0.381057
1,0.1,1.801518,5.612488,2.369069,0.379387
2,0.3,1.753382,5.707622,2.389063,0.368867
3,0.0,1.852657,5.752936,2.398528,0.363857
4,0.4,1.754914,5.943204,2.437869,0.342817
5,0.5,1.776011,6.304128,2.510802,0.302907
6,0.6,1.813238,6.790395,2.605839,0.249137
7,0.7,1.871531,7.402004,2.720662,0.181507
8,0.8,1.951364,8.138957,2.852886,0.100017
9,0.9,2.050601,9.001252,3.000209,0.004667


Selected alpha: 0.2


# 18. Final Test Prediction

The validation-selected α is applied once to the untouched test predictions.

In [18]:
y_test_final_log = (
    (1.0 - best_alpha) * y_test_main_log
    + best_alpha * y_test_retrieval_log
)

y_test_pred_usd = np.maximum(
    np.expm1(y_test_final_log),
    0.0
)

print("Final alpha:", best_alpha)


Final alpha: 0.2


# 19. Final Metrics — Required Table

This cell produces the complete requested metric table: MAE_log, MSE_log, RMSE_log, R2_log, MAE_USD, MSE_USD, RMSE_USD, R2_USD, RMSLE, and Training_Time_Seconds.

In [19]:
MAE_log = mean_absolute_error(y_test_log, y_test_final_log)
MSE_log = mean_squared_error(y_test_log, y_test_final_log)
RMSE_log = np.sqrt(MSE_log)
R2_log = r2_score(y_test_log, y_test_final_log)

MAE_USD = mean_absolute_error(y_test_usd, y_test_pred_usd)
MSE_USD = mean_squared_error(y_test_usd, y_test_pred_usd)
RMSE_USD = np.sqrt(MSE_USD)
R2_USD = r2_score(y_test_usd, y_test_pred_usd)

RMSLE = np.sqrt(
    mean_squared_log_error(
        np.maximum(y_test_usd, 0.0),
        np.maximum(y_test_pred_usd, 0.0)
    )
)

total_training_time = (
    stack_base_training_time
    + meta_training_time
    + mlp_training_time
)

final_metrics = pd.DataFrame([{
    "MAE_log": MAE_log,
    "MSE_log": MSE_log,
    "RMSE_log": RMSE_log,
    "R2_log": R2_log,
    "MAE_USD": MAE_USD,
    "MSE_USD": MSE_USD,
    "RMSE_USD": RMSE_USD,
    "R2_USD": R2_USD,
    "RMSLE": RMSLE,
    "Training_Time_Seconds": total_training_time
}])

display(final_metrics)


,MAE_log,MSE_log,RMSE_log,R2_log,MAE_USD,MSE_USD,RMSE_USD,R2_USD,RMSLE,Training_Time_Seconds
0,1.800568,5.735049,2.394796,0.389547,16694.540755,1.120057e+10,105832.762197,0.050919,2.394796,246.636227


# 20. Compare Base Models, Stack, MLP, Retrieval, and Hybrid

This diagnostic comparison makes it possible to see whether the stacking stage itself is useful and whether retrieval improves the MLP.

In [20]:
diagnostic_rows = []

for name, pred in [
    ("Random Forest", stack_test[:, model_names.index("random_forest")]),
    ("Extra Trees", stack_test[:, model_names.index("extra_trees")]),
    ("XGBoost", stack_test[:, model_names.index("xgboost")]),
    ("CatBoost", stack_test[:, model_names.index("catboost")]),
    ("SVR", stack_test[:, model_names.index("svr")]),
    ("Stack Meta-Learner", meta_test_pred),
    ("MLP Main", y_test_main_log),
    ("Soft Retrieval", y_test_retrieval_log),
    ("Final Hybrid", y_test_final_log)
]:
    diagnostic_rows.append({
        "Model": name,
        "MAE_log": mean_absolute_error(y_test_log, pred),
        "MSE_log": mean_squared_error(y_test_log, pred),
        "RMSE_log": np.sqrt(mean_squared_error(y_test_log, pred)),
        "R2_log": r2_score(y_test_log, pred)
    })

diagnostic_results = pd.DataFrame(diagnostic_rows)
display(diagnostic_results)


,Model,MAE_log,MSE_log,RMSE_log,R2_log
0,Random Forest,1.836601,5.916361,2.432357,0.370248
1,Extra Trees,1.885611,6.202540,2.490490,0.339786
2,XGBoost,1.698488,5.253349,2.292019,0.440820
3,CatBoost,1.747223,5.510708,2.347490,0.413426
4,SVR,1.922037,7.610765,2.758762,0.189891
5,Stack Meta-Learner,1.701223,5.256302,2.292663,0.440506
6,MLP Main,1.891725,5.908444,2.430729,0.371090
7,Soft Retrieval,2.231807,10.202226,3.194092,-0.085951
8,Final Hybrid,1.800568,5.735049,2.394796,0.389547


# 21. Compare Against the Existing Tuned-XGBoost Baseline

Your established tuned-XGBoost baseline is RMSE_log = 2.236051. It is used only as an external benchmark and is not used in this Method 3 pipeline.

In [21]:
BASELINE_RMSE_LOG = 2.236051

print(f"Tuned XGBoost baseline RMSE_log: {BASELINE_RMSE_LOG:.6f}")
print(f"Stacking Method 3 RMSE_log:       {RMSE_log:.6f}")
print(f"Difference:                        {RMSE_log - BASELINE_RMSE_LOG:+.6f}")
print("Benchmark beaten:", RMSE_log < BASELINE_RMSE_LOG)


Tuned XGBoost baseline RMSE_log: 2.236051
Stacking Method 3 RMSE_log:       2.394796
Difference:                        +0.158745
Benchmark beaten: False


# 22. Save Final Results and Artifacts

The complete metric table, α search, diagnostics, predictions, preprocessing, stack models, meta-learner, random projection, PCA, MLP, and retrieval configuration are saved to the checkpoint directory.

In [22]:
metrics_path = CHECKPOINT_DIR / "method3_stacking_final_metrics.csv"
alpha_path = CHECKPOINT_DIR / "method3_stacking_alpha_validation.csv"
diagnostic_path = CHECKPOINT_DIR / "method3_stacking_diagnostics.csv"
predictions_path = CHECKPOINT_DIR / "method3_stacking_test_predictions.csv"
artifacts_path = CHECKPOINT_DIR / "method3_stacking_artifacts.joblib"
metadata_path = CHECKPOINT_DIR / "method3_stacking_metadata.json"

final_metrics.to_csv(metrics_path, index=False)
alpha_validation_results.to_csv(alpha_path, index=False)
diagnostic_results.to_csv(diagnostic_path, index=False)

predictions = test_df.copy()
predictions["y_pred_log_main"] = y_test_main_log
predictions["y_pred_log_retrieval"] = y_test_retrieval_log
predictions["y_pred_log_final"] = y_test_final_log
predictions["y_pred_usd_final"] = y_test_pred_usd
predictions["selected_alpha"] = best_alpha

for i, name in enumerate(model_names):
    predictions[f"stack_{name}_prediction"] = stack_test[:, i]

predictions["stack_meta_prediction"] = meta_test_pred
predictions.to_csv(predictions_path, index=False)

joblib.dump({
    "robust_preprocessor": robust_preprocessor,
    "base_models": fitted_base_models,
    "meta_learner": meta_learner,
    "leaf_encoder": leaf_encoder,
    "tree_model_names": TREE_MODEL_NAMES,
    "Omega": Omega,
    "pca": pca,
    "norm_mean": norm_mean,
    "norm_std": norm_std,
    "mlp": mlp,
    "feature_columns": FEATURE_COLUMNS,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "best_alpha": best_alpha,
    "retrieval_k": RETRIEVAL_K,
    "retrieval_temperature": RETRIEVAL_TEMPERATURE,
    "random_seed": RANDOM_SEED
}, artifacts_path)

metadata = {
    "method": "Tree leaf embeddings (RF + ExtraTrees + XGBoost + CatBoost) + heterogeneous stacking (RF + ExtraTrees + XGBoost + CatBoost + SVR) -> RandomReLU -> PCA -> Normalization -> MLP -> Soft Retrieval",
    "target_log": TARGET_LOG,
    "target_usd": TARGET_USD,
    "random_seed": RANDOM_SEED,
    "stack_models": model_names,
    "stack_folds": N_STACK_FOLDS,
    "random_features": RANDOM_FEATURES,
    "main_dimension": MAIN_DIM,
    "mlp_hidden_layers": list(MLP_HIDDEN_LAYERS),
    "retrieval_k": RETRIEVAL_K,
    "retrieval_temperature": RETRIEVAL_TEMPERATURE,
    "alpha_grid": ALPHA_GRID,
    "selected_alpha": best_alpha,
    "benchmark_rmse_log": BASELINE_RMSE_LOG,
    "final_rmse_log": float(RMSE_log),
    "benchmark_beaten": bool(RMSE_log < BASELINE_RMSE_LOG),
    "test_used_for_alpha_selection": False,
    "retrieval_context": "model-training split only",
    "stacking_oof": True,
    "base_training_times_seconds": base_training_times,
    "stack_training_time_seconds": float(stack_base_training_time),
    "meta_training_time_seconds": float(meta_training_time),
    "mlp_training_time_seconds": float(mlp_training_time),
    "total_training_time_seconds": float(total_training_time)
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
for p in [
    metrics_path,
    alpha_path,
    diagnostic_path,
    predictions_path,
    artifacts_path,
    metadata_path
]:
    print(" ", p)


Saved:
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_4_v1\checkpoints\method3_stacking_final_metrics.csv
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_4_v1\checkpoints\method3_stacking_alpha_validation.csv
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_4_v1\checkpoints\method3_stacking_diagnostics.csv
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_4_v1\checkpoints\method3_stacking_test_predictions.csv
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_4_v1\checkpoints\method3_stacking_artifacts.joblib
  E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_4_v1\checkpoints\method3_stacking_metadata.json


# 23. Final Results

This is the final output cell for Method 3. Use this metric table for comparison with the earlier methods.

In [23]:
print("=" * 100)
print("METHOD 3 — STACKING + MLP + SOFT RETRIEVAL")
print("=" * 100)

display(final_metrics)

print(f"Selected alpha: {best_alpha}")
print(f"RMSE_log:       {RMSE_log:.6f}")
print(f"Baseline:       {BASELINE_RMSE_LOG:.6f}")
print(f"Beats baseline: {RMSE_log < BASELINE_RMSE_LOG}")
print("=" * 100)


METHOD 3 — STACKING + MLP + SOFT RETRIEVAL


,MAE_log,MSE_log,RMSE_log,R2_log,MAE_USD,MSE_USD,RMSE_USD,R2_USD,RMSLE,Training_Time_Seconds
0,1.800568,5.735049,2.394796,0.389547,16694.540755,1.120057e+10,105832.762197,0.050919,2.394796,246.636227


Selected alpha: 0.2
RMSE_log:       2.394796
Baseline:       2.236051
Beats baseline: False
